**End-to-End Machine Learning Pipeline for Life Cycle Assessment (LCA) Impact Prediction**

This repository contains an end-to-end Machine Learning pipeline specifically designed to bridge the gap between Machine Learning (ML) and Life Cycle Assessment (LCA).

The goal is to automatically test, optimize, and evaluate various regression models to accurately predict environmental impacts. Since LCA studies inherently deal with multiple impact categories at once (e.g., climate change, acidification, freshwater ecotoxicity), this script is built to handle multiple target variables simultaneously, making it much easier to run robust analyses on complex inventory datasets.


**1- What does this code do?**

In short, the script takes your .csv dataset and does all the heavy lifting for you:

*  Trains 13 predictive models, ranging from basic linear regressions to advanced tree-based algorithms (XGBoost, CatBoost, Random Forest) and Neural Networks.
*   Optimizes hyperparameters automatically using the Optuna framework.
*   Performs robust validation using Stratified K-Fold (10 folds) with multiple repeats.
*   Calculates detailed metrics, including R² (train and test), MSE, and an overfitting index.
*   Generates automated plots, including performance comparisons, trade-off charts (Computational Time vs. R²), and—crucially for environmental studies—feature explainability using SHAP.
*   Exports ready-to-use data, generating CSV spreadsheets with metric summaries and individual row-by-row predictions for further analysis.



**2- How to use it (Step-by-Step)**

The code is modularized so you don't have to mess with the core logic. All dataset adaptations happen in a single place: *SECTION 2 (USER CONFIGURATION SECTION)*.

You only need to change four variables:

*   ARQUIVO_CSV: The file path to your dataset.
*  IDX_INPUT_INICIO and IDX_INPUT_FIM: The numerical index range for your input columns (your inventory/predictive features).
*   IDX_LABEL_INICIO and IDX_LABEL_FIM: The numerical index range for your target columns (the environmental impact categories).
*  N_REPEATS: How many times the cross-validation process will be repeated.

Once that's set, just run the code.


**3- Understanding the Code Structure (Section by Section):**

If you want to explore or tweak the pipeline, it is organized into 9 main blocks. Here is what happens in each one:

**Section 1: Import Libraries & Setup**

Where the tools are loaded. We use scikit-learn for most models and metrics, optuna for smart hyperparameter tuning, and shap to open the models' "black box" and understand which inventory features actually drive the environmental impacts.

**Section 2: User Configuration**

The only area requiring manual input. It's the pipeline's control panel, where we define data location and column splits.

**Sections 3 & 4: Data Preparation & Integrity Check**

Data formatting (like comma vs. dot decimals) often causes silent crashes. This step reads the data as text, standardizes decimals, and forces numerical conversion. Then, it prints a quick sanity check to the console (means, minimums, maximums) to ensure the data was read correctly before you spend hours training models.

**Section 5: Model Definitions**

Here we define our algorithm "arsenal". We instantiate 13 models along with their hyperparameter grids. Optuna tests combinations within these boundaries (like tree depth for XGBoost or learning rate for Gradient Boosting) to find the best version of each algorithm.

**Section 6: Main Training Loop**

This is the core engine. For each target variable (impact category) in your dataset:

- The data is split into folds.
- For each fold, data is standardized (StandardScaler).
- Optuna finds the best hyperparameters.
- The final model is trained.
- Predictions are made and reverted to their original scale.
- Error metrics and execution times are recorded.

**Section 7: Results Consolidation**

After training all models for a specific impact category, the code calculates the mean and standard deviation across all folds and repeats. It outputs two .csv files (a metric summary and a detailed row-by-row prediction log) and prints tabulated data to the console, ready to be pasted directly into Excel or any statistical software.

**Section 8: Visualizations & SHAP**

In this phase, the code generates visual intelligence for each target:

- Bar Chart: Shows which model achieved the highest average R².
- Scatter Plot (Actual vs. Predicted): Uses the last validation fold to visually show model accuracy.
- Trade-off Plot (Time vs. R²): Crucial for decision-making. It plots model performance against processing time, helping you decide if a computationally heavy model is really worth the wait compared to a faster, highly efficient one.
- SHAP Summary Plot: Takes the best model of the run and plots the feature importance. If the best model is tree-based (which is super fast), it uses TreeExplainer. If it's a complex model (like SVR), it uses K-Means sampling to run KernelExplainer without freezing your machine. This is vital in LCA studies to explain why a certain impact is high or low.

**Section 9: Global Summary**

Once all impact categories are processed, the pipeline wraps up by plotting a final, massive scatter plot. It calculates the global average performance (R² and Time) of each model across all evaluated targets. This helps identify the absolute "champion" algorithm for your specific LCA database, regardless of the specific impact category it's trying to predict.

In [ ]:
# ==============================================================================
# SECTION 1: IMPORT LIBRARIES & SETUP
# ==============================================================================
!pip install optuna xgboost catboost shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import optuna
import warnings
import os
import shap
import ast

# --- Google Colab Setup ---
from google.colab import drive

# --- Pandas Display Configuration ---
# Prevents scientific notation and forces decimal display up to 35 places
pd.set_option('display.float_format', lambda x: '%.35f' % x)
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Prevent cell content truncation

# Sklearn Libraries
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression Models
from sklearn.linear_model import BayesianRidge, LinearRegression, ElasticNet, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

warnings.filterwarnings("ignore")



# ==============================================================================
# SECTION 2: ⚠️⚠️⚠️ ACTION REQUIRED: USER CONFIGURATION SECTION ⚠️⚠️⚠️
# ==============================================================================

# 1. CSV FILE PATH (Google Drive Path)
print("\nMounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

CSV_FILE = 'Your information'

# 2. INPUT COLUMNS RANGE (Index start and end for your predictive features = LCI information)
IDX_INPUT_START = 'Your information'
IDX_INPUT_END   = 'Your information'

# 3. TARGET COLUMNS RANGE (Index start and end for your target variables = impact categories)
IDX_LABEL_START = 'Your information'
IDX_LABEL_END   = 'Your information'

# 4. NUMBER OF REPETITIONS (e.g., 3 repeats x 10 folds = 30 tests per model)
N_REPEATS = 'Your information'

# ==============================================================================
# 👆👆👆 END OF USER CONFIGURATION 👆👆👆
# ==============================================================================

try:
    dataset_name = os.path.splitext(os.path.basename(CSV_FILE))[0]
    output_dir = os.path.dirname(CSV_FILE)
except:
    dataset_name = "Analysis_Dataset"
    output_dir = "."

print(f"Processing dataset: {dataset_name}")
print(f"Output directory for results: {output_dir}")



# ==============================================================================
# SECTION 3: DATA PREPARATION & SAFE LOADING
# ==============================================================================
print(f"\nLoading dataset from: {CSV_FILE}")

# Read everything as string first to avoid Pandas point/comma confusion
df_dataset = pd.read_csv(CSV_FILE, sep=';', dtype=str)

# Replace commas with dots across the dataframe to ensure standard Python float format
df_dataset = df_dataset.apply(lambda x: x.str.replace(',', '.', regex=False) if x.dtype == "object" else x)

# Convert to numeric (text becomes NaN, valid strings become floats)
cols_to_convert = df_dataset.columns[IDX_INPUT_START : IDX_LABEL_END]
df_dataset[cols_to_convert] = df_dataset[cols_to_convert].apply(pd.to_numeric, errors='coerce')

input_columns = df_dataset.columns[IDX_INPUT_START : IDX_INPUT_END]
label_columns = df_dataset.columns[IDX_LABEL_START : IDX_LABEL_END]
id_column_name = df_dataset.columns[0]



# ==============================================================================
# SECTION 4: DATA INTEGRITY CHECK
# ==============================================================================
print("\n" + "="*80)
print(f"IMPORTED DATA CHECK ({len(label_columns)} TARGETS FOUND)")
print("="*80)

for i, col_name in enumerate(label_columns):
    col_verification = df_dataset[col_name]
    print(f"\n[{i+1}/{len(label_columns)}] Column Name: {col_name}")
    print("-" * 40)
    print("First 5 rows:")
    print(col_verification.head())
    print("-" * 40)
    print(f"Minimum: {col_verification.min()}")
    print(f"Maximum: {col_verification.max()}")
    print(f"Mean:    {col_verification.mean()}")
    print("." * 80)

print("\n" + "="*80 + "\n")

# Optional: Outlier Removal Function (Currently bypassed to keep raw data integrity)
def remove_outliers_from_labels(df, label_columns, threshold=1.5, show_plots=False):
    df_clean = df.copy()
    return df_clean, {}



# ==============================================================================
# SECTION 5: MODEL DEFINITIONS & HYPERPARAMETERS
# ==============================================================================
regression_models = [
    ("Bayes", BayesianRidge(), {"regressor__alpha_1": [1e-6, 1e-4, 1e-2], "regressor__lambda_1": [1e-6, 1e-4, 1e-2]}),
    ("KNN", KNeighborsRegressor(), {"regressor__n_neighbors": [5, 7, 10, 15, 20], "regressor__weights": ["uniform", "distance"]}),
    ("LR", LinearRegression(), {}),
    ("E_Net", ElasticNet(), {"regressor__alpha": [0.01, 0.1, 1.0, 10.0], "regressor__l1_ratio": [0.1, 0.5, 0.9]}),
    ("SGD", SGDRegressor(), {"regressor__alpha": [1e-4, 1e-3, 0.01], "regressor__penalty": ['l2', 'l1', 'elasticnet']}),
    ("Kernel", KernelRidge(), {"regressor__alpha": [0.01, 0.1, 1.0]}),
    ("DT", DecisionTreeRegressor(), {"regressor__max_depth": [3, 5, 7, 10], "regressor__min_samples_leaf": [5, 10, 20]}),
    ("RF", RandomForestRegressor(), {"regressor__n_estimators": [50, 100], "regressor__max_depth": [3, 5, 7, 10], "regressor__min_samples_leaf": [2, 5, 10]}),
    ("GBM", GradientBoostingRegressor(), {"regressor__n_estimators": [50, 100], "regressor__learning_rate": [0.01, 0.05, 0.1], "regressor__max_depth": [3, 5], "regressor__subsample": [0.7, 0.8]}),
    ("XGBoost", XGBRegressor(), {"regressor__n_estimators": [50, 100], "regressor__max_depth": [3, 4, 5], "regressor__reg_alpha": [0.1, 1], "regressor__reg_lambda": [1, 5]}),
    ("CatBoost", CatBoostRegressor(verbose=0), {}),
    ("ANN", MLPRegressor(max_iter=1000), {"regressor__hidden_layer_sizes": [(10,), (20,), (50,), (10, 10)], "regressor__alpha": [0.001, 0.01]}),
    ("SVM", SVR(), {"regressor__C": [0.1, 1, 10], "regressor__gamma": ["scale", "auto"]})
]



# ==============================================================================
# SECTION 6: MAIN TRAINING LOOP & EVALUATION
# ==============================================================================

global_r2_time_stats = []

for label_column in label_columns:
    start_total_target = time.time()

    # Clean target name for file saving
    target_clean = "".join([c for c in label_column if c.isalnum() or c in (' ', '_')]).strip()

    print(f"\n{'='*60}\n=== Processing: {label_column} ===\n{'='*60}")

    df_base = df_dataset.dropna(subset=[label_column]).reset_index(drop=True)
    groups = df_base[id_column_name].astype(str).apply(lambda x: x[:2])

    print("Samples per database (Total):")
    print(groups.value_counts().to_string())
    print("-" * 30)

    metrics_results = []
    predictions_results = []

    last_y_test_real = None
    last_y_pred_real = None

    for i_repeat in range(N_REPEATS):
        current_seed = 42 + i_repeat
        print(f"\n>>> REPETITION {i_repeat + 1}/{N_REPEATS} (Seed: {current_seed}) <<<")

        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=current_seed)
        current_fold = 0

        for train_index, test_index in skf.split(df_base, groups):
            current_fold += 1

            df_train_raw = df_base.iloc[train_index]
            df_test_raw = df_base.iloc[test_index]

            # --- KEEP OUTLIERS ---
            df_train_clean = df_train_raw.copy()
            df_test_clean = df_test_raw.copy()

            if len(df_train_clean) == 0 or len(df_test_clean) == 0: continue

            train_dist = df_train_clean[id_column_name].astype(str).apply(lambda x: x[:2]).value_counts().to_dict()
            test_dist = df_test_clean[id_column_name].astype(str).apply(lambda x: x[:2]).value_counts().to_dict()

            if current_fold == 1:
                print(f"  Fold 1 Example: Train {train_dist} | Test {test_dist}")
                print(f"  Training...", end=" ")

            X_train = df_train_clean[input_columns]
            y_train = df_train_clean[label_column]
            X_test = df_test_clean[input_columns]
            y_test = df_test_clean[label_column]

            ids_test = df_test_clean[id_column_name].values

            # Normalization
            scaler_labels = StandardScaler()
            y_train_scaled = scaler_labels.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            y_test_scaled = scaler_labels.transform(y_test.values.reshape(-1, 1)).ravel()

            for model_name, model, params in regression_models:
                if current_fold == 1: print(f".", end="")
                start_time = time.time()
                try:
                    pipeline = Pipeline([
                        ("scaler", StandardScaler()),
                        ("regressor", model)
                    ])

                    best_params_fold = "Default/None"

                    if params:
                        optuna.logging.set_verbosity(optuna.logging.ERROR)
                        def objective(trial):
                            optuna_params = {}
                            for param_name, values in params.items():
                                if values is not None:
                                    if isinstance(values[0], int):
                                        optuna_params[param_name] = trial.suggest_int(param_name, min(values), max(values))
                                    elif isinstance(values[0], float):
                                        optuna_params[param_name] = trial.suggest_float(param_name, min(values), max(values))
                                    elif isinstance(values[0], str):
                                        optuna_params[param_name] = trial.suggest_categorical(param_name, values)
                            pipeline.set_params(**optuna_params)
                            score = cross_val_score(pipeline, X_train, y_train_scaled, cv=3, scoring='r2')
                            return np.mean(score)

                        study = optuna.create_study(direction="maximize")
                        study.optimize(objective, n_trials=5, timeout=60)
                        best_params_fold = study.best_params
                        pipeline.set_params(**best_params_fold)
                        pipeline.fit(X_train, y_train_scaled)
                        best_model = pipeline
                    else:
                        pipeline.fit(X_train, y_train_scaled)
                        best_model = pipeline

                    y_pred_scaled = best_model.predict(X_test)

                    # RECOVERING REAL VALUES
                    y_test_real = y_test.values
                    y_pred_real = scaler_labels.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

                    # y_train_real (For overfitting calculation)
                    y_train_pred_scaled = best_model.predict(X_train)
                    y_train_real = y_train.values
                    y_train_pred_real = scaler_labels.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()

                    if model_name in ["CatBoost", "XGBoost", "RF"]:
                         last_y_test_real = y_test_real
                         last_y_pred_real = y_pred_real

                    r2_test = r2_score(y_test_real, y_pred_real)
                    mse_test = mean_squared_error(y_test_real, y_pred_real)
                    mse_train = mean_squared_error(y_train_real, y_train_pred_real)

                    overfitting_index = abs(mse_train - mse_test) / mse_test if mse_test != 0 else 0
                    total_time = time.time() - start_time

                    metrics_results.append({
                        "Repeat": i_repeat + 1,
                        "Fold": current_fold,
                        "Model": model_name,
                        "R2": r2_test,
                        "MSE_Test": mse_test,
                        "MSE_Train": mse_train,
                        "Overfitting": overfitting_index,
                        "Time_s": total_time,
                        "Best_Hyperparameters": str(best_params_fold),
                        "Train_Distribution": str(train_dist),
                        "Test_Distribution": str(test_dist)
                    })

                    for idx_sample, val_real, val_pred in zip(ids_test, y_test_real, y_pred_real):
                        predictions_results.append({
                            "Unique_Key": f"{dataset_name}-{target_clean}-{idx_sample}-{i_repeat+1}-{current_fold}-{model_name}",
                            "Database_Version": dataset_name,
                            "Target": target_clean,
                            "Sample_ID": idx_sample,
                            "Repeat": i_repeat + 1,
                            "Fold": current_fold,
                            "Model": model_name,
                            "Real_Value": val_real,
                            "Predicted_Value": val_pred
                        })

                except Exception as e:
                    pass



    # ==============================================================================
    # SECTION 7: RESULTS CONSOLIDATION & SAVING
    # ==============================================================================
    print(f"\n\nCalculating statistics and saving files...")

    end_total_target = time.time()
    total_duration = end_total_target - start_total_target
    print(f"\n[Timer] Total processing time for {label_column}: {total_duration/60:.2f} minutes ({total_duration:.1f}s)")

    df_detailed = pd.DataFrame(metrics_results).round(5)
    df_predictions = pd.DataFrame(predictions_results)

    if not df_detailed.empty:
        # Metrics Aggregation
        df_summary = df_detailed.groupby('Model').agg({
            'R2': ['mean', 'std', 'min', 'max'],
            'MSE_Test': ['mean', 'std'],
            'MSE_Train': ['mean', 'std'],
            'Overfitting': 'mean',
            'Time_s': 'mean'
        }).reset_index()

        # Rename columns to standard format
        df_summary.columns = [
            'Model',
            'R2_Mean', 'R2_Std', 'R2_Min', 'R2_Max',
            'MSE_Test_Mean', 'MSE_Test_Std',
            'MSE_Train_Mean', 'MSE_Train_Std',
            'Overfitting_Mean',
            'Time_Mean_s'
        ]
        df_summary = df_summary.sort_values(by='R2_Mean', ascending=False)

        # --- SAVE CSV FILES (English Standard Format) ---
        file_summary = os.path.join(output_dir, f"Results_COMPILATION_{dataset_name}_{target_clean}.csv")
        file_predictions = os.path.join(output_dir, f"Results_INDIVIDUALS_{dataset_name}_{target_clean}.csv")

        df_summary.to_csv(file_summary, sep=',', index=False, decimal='.')
        df_predictions.to_csv(file_predictions, sep=',', index=False, decimal='.')

        print(f"✅ Files successfully saved (Comma-separated, dot-decimal):")
        print(f"   -> {file_summary}")
        print(f"   -> {file_predictions}")

        # --- PRINT TABLES TO CONSOLE ---
        print(f"\n{'#'*60}")
        print(f"FINAL RESULTS - {label_column}")
        print(f"{'#'*60}")

        print("\n>>> SUMMARY TABLE (COPY THE TEXT BELOW AND PASTE INTO EXCEL) <<<")
        print(df_summary.to_csv(sep='\t', index=False, decimal='.'))

        print("\n>>> DETAILED TABLE (COPY THE TEXT BELOW AND PASTE INTO EXCEL) <<<")
        print(df_detailed.to_csv(sep='\t', index=False, decimal='.'))

        print(f"{'#'*60}\n")


    # ==============================================================================
    # SECTION 8: VISUALIZATIONS (R², TRADE-OFF, SHAP)
    # ==============================================================================
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_detailed, x='Model', y='R2', palette="viridis", errorbar="sd")
    plt.title(f"Model Performance (Test R²) - {label_column}")
    plt.ylabel("R² (Test)")
    plt.xlabel("Model")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    if last_y_test_real is not None:
        plt.figure(figsize=(10, 6))
        plt.scatter(range(len(last_y_test_real)), last_y_test_real, color='red', label='Actual', alpha=0.6)
        plt.scatter(range(len(last_y_pred_real)), last_y_pred_real, color='blue', label='Predicted', alpha=0.6)
        plt.title(f"Actual vs Predicted Visualization (Last Fold Sample) - {label_column}")
        plt.legend()
        plt.show()

    # --- Trade-off Chart: R2 vs Time ---
    if not df_detailed.empty:
        fig, axes = plt.subplots(1, 2, figsize=(20, 6))

        # Subplot 1: All Models
        sns.scatterplot(data=df_summary, x='Time_Mean_s', y='R2_Mean', hue='Model', s=150, palette='tab20', ax=axes[0])
        axes[0].set_title(f"Trade-off: Mean R² vs Processing Time (All Models) - {label_column}")
        axes[0].set_xlabel("Mean Processing Time (s)")
        axes[0].set_ylabel("Mean R² (Test)")
        axes[0].grid(True, linestyle='--', alpha=0.7)
        axes[0].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')

        # Subplot 2: R2 > 0 Models only
        df_summary_pos = df_summary[df_summary['R2_Mean'] > 0]

        if not df_summary_pos.empty:
            sns.scatterplot(data=df_summary_pos, x='Time_Mean_s', y='R2_Mean', hue='Model', s=150, palette='tab20', ax=axes[1])
            axes[1].set_title(f"Trade-off: Mean R² vs Processing Time (R² > 0) - {label_column}")
            axes[1].set_xlabel("Mean Processing Time (s)")
            axes[1].set_ylabel("Mean R² (Test)")
            axes[1].grid(True, linestyle='--', alpha=0.7)
            axes[1].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        else:
            axes[1].text(0.5, 0.5, 'No models with R² > 0', ha='center', va='center', fontsize=12)
            axes[1].set_axis_off()

        plt.tight_layout()
        plt.show()

        # Store data for the final global chart
        df_summary_temp = df_summary.copy()
        df_summary_temp['Category'] = label_column
        global_r2_time_stats.append(df_summary_temp[['Category', 'Model', 'R2_Mean', 'Time_Mean_s']])

    # --- Feature Importance (SHAP) ---
    if not df_detailed.empty:
        best_model_name = df_summary.iloc[0]['Model']
        print(f"\n--- Generating SHAP Analysis for the best model: {best_model_name} ---")

        model_results = df_detailed[df_detailed['Model'] == best_model_name]

        best_row = model_results.loc[model_results['R2'].idxmax()]

        str_params = best_row['Best_Hyperparameters']

        shap_model = None
        for m_name, mod, param in regression_models:
            if m_name == best_model_name:
                shap_model = mod
                break

        if shap_model is not None:
            try:
                # Apply Optuna parameters before fitting SHAP model
                if str_params != "Default/None":
                    dict_params = ast.literal_eval(str_params)
                    clean_params = {k.replace('regressor__', ''): v for k, v in dict_params.items()}
                    shap_model.set_params(**clean_params)

                shap_model.fit(X_train, y_train_scaled)

                tree_models = ["DT", "RF", "GBM", "XGBoost", "CatBoost"]

                if best_model_name in tree_models:
                    explainer = shap.TreeExplainer(shap_model)
                    shap_values = explainer.shap_values(X_test)

                    plt.figure(figsize=(10, 6))
                    plt.title(f"SHAP Summary Plot - {best_model_name} ({label_column})")
                    shap.summary_plot(shap_values, X_test, show=False)
                    plt.tight_layout()
                    plt.show()
                else:
                    print(f"Using KernelExplainer for {best_model_name} (calculating on a safe sample...)")
                    background = shap.kmeans(X_train, 10)
                    explainer = shap.KernelExplainer(shap_model.predict, background)

                    X_test_sample = X_test[:40]
                    shap_values = explainer.shap_values(X_test_sample)

                    plt.figure(figsize=(10, 6))
                    plt.title(f"SHAP Summary Plot (Sample) - {best_model_name} ({label_column})")
                    shap.summary_plot(shap_values, X_test_sample, show=False)
                    plt.tight_layout()
                    plt.show()

            except Exception as e:
                print(f"Could not generate SHAP for {best_model_name}: {e}")



# ==============================================================================
# SECTION 9: GLOBAL SUMMARY
# ==============================================================================
if global_r2_time_stats:
    print("\n" + "="*80)
    print("GENERATING FINAL GLOBAL CHART: Mean R² vs Time (Average across all categories)")
    print("="*80)

    df_global_stats = pd.concat(global_r2_time_stats)

    df_final_agg = df_global_stats.groupby('Model').agg({
        'R2_Mean': 'mean',
        'Time_Mean_s': 'mean'
    }).reset_index()

    plt.figure(figsize=(14, 8))
    sns.scatterplot(data=df_final_agg, x='Time_Mean_s', y='R2_Mean', hue='Model', s=200, palette='tab20', edgecolor='black')

    for _, row in df_final_agg.iterrows():
        plt.annotate(row['Model'], (row['Time_Mean_s'], row['R2_Mean']),
                     xytext=(8, 8), textcoords='offset points', fontsize=11, fontweight='bold')

    plt.title("Global Performance: Mean R² vs Mean Processing Time (All Categories)", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Mean Processing Time per Model (s)", fontsize=13)
    plt.ylabel("Global Mean R²", fontsize=13)
    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)

    plt.tight_layout()
    plt.show()


Mounting Google Drive...
Mounted at /content/drive
Processing dataset: data
Output directory for results: /content/drive/My Drive/Mestrado/0° Dados de inventário/1- Agricultura (testes 1 e 3)/2 - Total agrícolas (Teste 3)/Testes/1 - Teste final: após a revisão SPC/1 - Bases compiladas: após revisão SPC

Loading dataset from: /content/drive/My Drive/Mestrado/0° Dados de inventário/1- Agricultura (testes 1 e 3)/2 - Total agrícolas (Teste 3)/Testes/1 - Teste final: após a revisão SPC/1 - Bases compiladas: após revisão SPC/data.csv

IMPORTED DATA CHECK (3 TARGETS FOUND)

[1/3] Column Name: Impact 1
----------------------------------------
First 5 rows:
0   0.80000000000000004440892098500626162
1   0.90000000000000002220446049250313081
2   0.50000000000000000000000000000000000
3   0.29999999999999998889776975374843460
4   0.40000000000000002220446049250313081
Name: Impact 1, dtype: float64
----------------------------------------
Minimum: 0.1
Maximum: 1.0
Mean:    0.535
...................